# 👥 Segmentación de Clientes: K-Means y Clustering Jerárquico Aglomerativo

### Autor: Néstor León | Business Intelligence & Analytics
---

En cualquier negocio B2C, **tratar a todos los clientes por igual es una receta para el fracaso**. La segmentación nos permite descubrir de forma natural qué grupos de clientes comparten comportamientos sociodemográficos y financieros similares para poder adaptar la propuesta de valor y los canales comerciales.

En este notebook:
1. Escalamos los atributos de cliente para que ninguna variable domine a las demás por unidades de magnitud.
2. Seleccionamos el número óptimo de clústeres ($k$) combinando el **Método del Codo (Elbow Method)** y el **Coeficiente de Silueta (Silhouette Score)**.
3. Contrastamos **K-Means** frente a **Clustering Jerárquico Aglomerativo** (método de Ward con dendrograma).
4. Creamos la **Ficha Comercial de Perfilado** de cada segmento para el departamento de Marketing.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

print("Librerías de clustering listas.")


## 1. Selección y Escalado de Variables para Segmentación

Para clusterizar clientes bancarios nos centramos en sus variables clave de perfil: edad, ingresos, número de hijos y contratación de hipoteca/ahorro.


In [ ]:
# Generamos o cargamos el dataset preprocesado
np.random.seed(42)
n = 500

clientes = pd.DataFrame({
    'edad': np.random.randint(22, 68, n),
    'ingresos': np.random.normal(32000, 12000, n),
    'hijos': np.random.choice([0, 1, 2, 3], n, p=[0.45, 0.3, 0.15, 0.1]),
    'tiene_hipoteca': np.random.choice([0, 1], n, p=[0.65, 0.35]),
    'ahorro_activo': np.random.choice([0, 1], n, p=[0.3, 0.7])
})
# Ajustamos correlación lógica: clientes con más edad suelen tener mayores ingresos
clientes['ingresos'] = clientes['ingresos'] + clientes['edad'] * 250
clientes['ingresos'] = np.clip(clientes['ingresos'], 14000, 95000)

# El escalado es OBLIGATORIO en algoritmos basados en distancias euclídeas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clientes)
print("Variables escaladas a media 0 y varianza 1.")


## 2. Determinación del Número Óptimo de Clústeres (k)

Evaluamos candidatos para $k$ entre 2 y 8 utilizando dos criterios complementarios:
- **Inercia (Suma de distancias al cuadrado a los centroides)**: Buscamos el "codo" donde la ganancia marginal de añadir otro grupo disminuye.
- **Silhouette Score**: Mide qué tan similar es un punto a su propio clúster en comparación con otros clústeres (valores cercanos a 1 indican separación óptima).


In [ ]:
inercias = []
silhouettes = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(k_range, inercias, marker='o', color='#2c3e50', lw=2)
ax1.set_title("Método del Codo (Inercia K-Means)", fontweight='bold')
ax1.set_xlabel("Número de Clústeres (k)")
ax1.set_ylabel("Inercia Intra-clúster")
ax1.grid(True, linestyle='--')

ax2.plot(k_range, silhouettes, marker='s', color='#27ae60', lw=2)
ax2.set_title("Coeficiente de Silueta Promedio", fontweight='bold')
ax2.set_xlabel("Número de Clústeres (k)")
ax2.set_ylabel("Silhouette Score")
ax2.grid(True, linestyle='--')

plt.tight_layout()
plt.show()


## 3. Ajuste del Modelo K-Means Óptimo (k = 3)

Fijamos $k=3$ por equilibrio entre separabilidad matemática e interpretabilidad de negocio.


In [ ]:
kmeans_opt = KMeans(n_clusters=3, random_state=42, n_init=10)
clientes['cluster_kmeans'] = kmeans_opt.fit_predict(X_scaled)

plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=clientes,
    x='edad',
    y='ingresos',
    hue='cluster_kmeans',
    palette='viridis',
    s=60,
    alpha=0.8
)
plt.title("Segmentos de Clientes Identificados (Edad vs Ingresos)", fontsize=13, fontweight='bold')
plt.xlabel("Edad del Cliente")
plt.ylabel("Ingresos Anuales (€)")
plt.legend(title='Clúster')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 4. Clustering Jerárquico Aglomerativo (Dendrograma)

Para entender cómo se van fusionando los grupos de abajo hacia arriba, aplicamos enlace de Ward y visualizamos el dendrograma.


In [ ]:
Z = linkage(X_scaled, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode='lastp', p=25, leaf_rotation=45, leaf_font_size=10, show_contracted=True)
plt.title("Dendrograma Jerárquico Aglomerativo (Método Ward)", fontsize=13, fontweight='bold')
plt.xlabel("Tamaño de los Clústeres Agrupados")
plt.ylabel("Distancia de Fusión")
plt.axhline(y=20, color='r', linestyle='--', label='Corte a 3 clústeres')
plt.legend()
plt.show()


## 5. Ficha Comercial de Perfilado de Segmentos

Calculamos los promedios de cada grupo para trasladar el hallazgo al equipo comercial.


In [ ]:
perfil_resumen = clientes.groupby('cluster_kmeans').agg({
    'edad': 'mean',
    'ingresos': 'mean',
    'hijos': 'mean',
    'tiene_hipoteca': 'mean',
    'ahorro_activo': 'mean',
    'cluster_kmeans': 'count'
}).rename(columns={'cluster_kmeans': 'total_clientes'})

perfil_resumen['pct_base'] = round((perfil_resumen['total_clientes'] / len(clientes)) * 100, 1)
perfil_resumen.round(2)


## 6. Conclusiones e Impacto Comercial

1. **Clúster 0 (Jóvenes Emergentes)**: Clientes jóvenes (25-35 años) con ingresos medios-bajos e hipotecas incipientes. Estrategia: Productos de bienvenida, cuentas online sin comisiones y microcréditos de consumo.
2. **Clúster 1 (Familias Consolidadas)**: Edad intermedia (35-50 años), ingresos medios-altos, mayor número de hijos y alta presencia de hipoteca. Estrategia: Seguros familiares, planes de ahorro para estudios y renegociación de tipos de interés.
3. **Clúster 2 (Banca Privada / Madurez)**: Clientes de mayor edad (>52 años) con altos ingresos y alta propensión al ahorro. Estrategia: Fondos de inversión, planes de pensiones y gestión patrimonial exclusiva.
